# Multi-instrument Example

This notebook contains an example showing how to search for a transient over the sky using multiple instruments. For the purpose of simplicity, we'll use groups of detectors on the Fermi GBM spacecraft as two separate instruments but in practice you can use instruments across any spacecraft that provide data and responses.

We'll begin by defining our two separate instrument configurations:

In [1]:
import configuration

from gdt.missions.fermi.gbm.detectors import GbmDetectors

nai_right = {det: {'channel_edges': [0, 8, 20, 33, 51, 85, 106, 127, 128], 'search_channels': [1, 2, 3, 4, 5, 6]} for det in ['n0', 'n1', 'n2', 'n3', 'n4', 'n5']}
instrument1 = configuration.InstrumentConfiguration('instrument1', nai_right)

nai_left = {det: {'channel_edges': [0, 8, 20, 33, 51, 85, 106, 127, 128], 'search_channels': [1, 2, 3, 4, 5, 6]} for det in ['n6', 'n7', 'n8', 'n9', 'na', 'nb']}
instrument2 = configuration.InstrumentConfiguration('instrument2', nai_left)

Next we'll download data around a trigger time to search

In [2]:
import glob

from gdt.missions.fermi.time import Time
from gdt.missions.fermi.gbm.finders import ContinuousFinder

trigtime = Time(524666471.413, format='fermi')
finder = ContinuousFinder(trigtime)

# download TTE
finder.get_tte("data/gbm", dets=instrument1['detector_names'] + instrument2['detector_names'])
tte_files = sorted(glob.glob('data/gbm/glg_tte_n?_170817_12z_v??.fit.gz'))

# download spacecraft position history
finder.get_poshist('data/gbm')
poshist_file = glob.glob('data/gbm/glg_poshist_all_170817_v??.fit')[-1]

# summary of local files
print("Downloaded TTE files:")
for file in tte_files:
    print(f"  {file}")
print(f"\nDownloaded PosHist file: {poshist_file}")

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

Downloaded TTE files:
  data/gbm/glg_tte_n0_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n1_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n2_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n3_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n4_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n5_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n6_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n7_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n8_170817_12z_v00.fit.gz
  data/gbm/glg_tte_n9_170817_12z_v00.fit.gz
  data/gbm/glg_tte_na_170817_12z_v00.fit.gz
  data/gbm/glg_tte_nb_170817_12z_v00.fit.gz

Downloaded PosHist file: data/gbm/glg_poshist_all_170817_v01.fit


Let's also define the search settings we'd like to use:

In [3]:
search_config = configuration.SearchConfiguration(instruments=[instrument1, instrument2], skygrid_resolution=5.0)
search_config.settings.update({
    'win_width': 60,           # Window around T0 to search: search is run between T0-width/2 and T0+width/2 seconds
    'min_dur': 0.512,          # Minimum search duration in seconds
    'max_dur': 0.512,          # Maximum search duration in seconds
    'min_step': 0.512,         # Minimum phase shift in seconds
    'num_steps': 1,            # Number of phase shifts per duration
    'min_loglr': 5,            # Minimum loglr to produce plots
    'bkgd_range': [-500, 500], # background fit interval in seconds
    'bkgd_window': 125,        # background average window in seconds
    'data_range': [-30.512, 30.512]}) # data range to pre-bin in temporal bins

and setup a search object

In [4]:
import utils
import search

skygrid = utils.SkyGrid(search_config['skygrid_resolution'])
targeted_search = search.TargetedSearch(search_config, skygrid)

We can now load the data + responses for each instrument and add them to the search object

In [5]:
import data
import response
import numpy as np

from gdt.core.collection import DataCollection
from gdt.core.binning.binned import rebin_by_edge_index
from gdt.core.binning.unbinned import bin_by_time
from gdt.core.background.fitter import BackgroundFitter
from gdt.core.background.unbinned import NaivePoisson
from gdt.missions.fermi.gbm.tte import GbmTte
from gdt.missions.fermi.gbm.poshist import GbmPosHist

print("")
for inst_config in [instrument1, instrument2]:

    print(f"  {inst_config['name']}")
    print(f"    Opening TTE")
    tte_data = []
    for det in inst_config['detector_names']:
        path = f"data/gbm/glg_tte_{det}_170817_12z_v00.fit.gz"
        tte = utils.update_tte_trigtime(GbmTte.open(path), trigtime.fermi)
        tte = tte.rebin_energy(rebin_by_edge_index, np.array(inst_config['detectors'][det]['channel_edges']))
        tte_data.append(tte)
    ttes = DataCollection.from_list(tte_data, names=inst_config['detector_names'])

    print(f"    Opening PosHist")
    poshist = GbmPosHist.open("data/gbm/glg_poshist_all_170817_v01.fit")
    spacecraft_frames = poshist.get_spacecraft_frame()

    print(f"    Opening Response")
    rsp = response.GbmResponse(inst_config['detector_names'], utils.SkyGrid(5.0), 'templates/GBM',
                               spacecraft_frames, trigtime.fermi, templates=[0, 1, 2])

    print(f"    Fitting background")
    backfitters = DataCollection.from_list(
        [BackgroundFitter.from_tte(tte.slice_time(search_config['bkgd_range']), NaivePoisson) for tte in ttes],
         names=inst_config['detector_names'])
    backfitters.fit(window_width=search_config['bkgd_window'], fast=True)

    goodness_of_fit = DataCollection.from_list(
        [data.FitStatus(len(edges) - 1) for det, edges in inst_config['channel_edges'].items()],
        names=inst_config['detector_names'])

    targeted_search.add_instrument(inst_config['name'], ttes, backfitters, goodness_of_fit, rsp)


  instrument1
    Opening TTE
    Opening PosHist
    Opening Response
    Fitting background
  instrument2
    Opening TTE
    Opening PosHist
    Opening Response
    Fitting background


Now we can compute the likelihood in a single timebin for each point in the 5 degree skygrid used for the spatial search of source locations on the sky.

This will take a while because there are 1634 positions in the 5 degree skygrid. For each one we need to compute the time offset between the two instruments, collect data, and rotate the reponse of the second instrument into the frame of the first instrument.

In [6]:
import time

wallt = time.time()
targeted_search.calculate_likelihood(-0.256, 0.256, sky_mask=True)
print(f"\nCompleted likelihood in {time.time() - wallt:.1f} sec")


Completed likelihood in 66.5 sec


We can display the best-fit location in the frame of the first instrument as well as the log likelihood ratio marginalized over the skygrid using member values from the `targeted_search` object:

In [7]:
az_max, zen_max = targeted_search.like_points[:, targeted_search.like.max_location]
print("\nBest-fit (az %.1f deg, zen %.1f deg) marginal llr %.2f" % (np.degrees(az_max), np.degrees(zen_max), targeted_search.like.marginal_llr))


Best-fit (az 25.0 deg, zen 90.0 deg) marginal llr 71.16


And we convert the best-fit location to Equatorial coordinates using the likelihood frame object

In [8]:
from astropy.coordinates import SkyCoord

coord = SkyCoord([az_max], [0.5 * np.pi - zen_max], frame=targeted_search.like_frame, unit='rad')

print("Equatorial Coord (ra %.1f deg, dec %.1f deg)" % (coord.icrs.ra.degree[0], coord.icrs.dec.degree[0]))

Equatorial Coord (ra 179.9 deg, dec -37.9 deg)


You may proceed to search additional timebins within this notebook, but it's better to run them separately on a multicore server given that it's computationally intensive. Alternatively, you can increase the resolution of the skygrid to speed things up.